## Key Findings and Next Steps

### Summary of Results

This notebook implemented the recommended next steps for improving the ONNX latency prediction model:

**1. Boosting Models:** 
- Trained LightGBM and XGBoost on log-transformed target (log of average_ms)
- Expected to show better precision than Random Forest baseline

**2. Feature Refinement:**
- Dropped extremely sparse features (≥99.9% zeros)
- Removed low-variance features
- Added binary presence flags for medium-sparse features (50-95% zeros)
- Reduced feature count while preserving predictive signal

**3. Error Analysis:**
- Analyzed error distribution across low/medium/high latency ranges
- Identified potential target-range bias
- Visualized residuals and relative error patterns

**4. Feature Importance:**
- Extracted feature importance from best model
- Compared contributions: dense vs sparse vs categorical features
- Identified top predictive features

**5. Model Artifacts:**
- Saved best model (LightGBM with refined features)
- Saved preprocessor and feature metadata
- Saved predictions for external validation
- Saved feature importance rankings

### Recommended Actions

1. **Compare Test Performance:** Check if boosting models improved within-10% and within-25% thresholds vs Random Forest baseline
2. **Investigate Sparse Flags:** Evaluate whether binary presence flags improved performance by capturing zero-vs-nonzero splits
3. **Target-Range Analysis:** Review error by latency range to identify systematic biases
4. **Feature Reduction:** Consider whether removing sparse/low-variance features helps generalization
5. **Hyperparameter Tuning:** Fine-tune LightGBM/XGBoost parameters using validation set performance
6. **Ensemble Methods:** Consider ensemble combinations of top-performing models

In [ ]:
# Save best model and metadata
artifacts_dir = Path.cwd() / "training" / "artifacts"
artifacts_dir.mkdir(parents=True, exist_ok=True)

# Save the best model (LightGBM with refined features)
model_path = artifacts_dir / "lgb_refined_model.pkl"
with open(model_path, 'wb') as f:
    pickle.dump(lgb_refined, f)

# Save preprocessor
preprocessor_path = artifacts_dir / "preprocess_refined.pkl"
with open(preprocessor_path, 'wb') as f:
    pickle.dump(preprocess_refined, f)

# Save feature lists
feature_metadata = {
    'original_feature_cols': feature_cols,
    'refined_numeric_features': refined_numeric_features,
    'categorical_features': categorical_features,
    'removed_sparse_cols': very_sparse_cols,
    'removed_low_var_cols': low_var_cols,
}

feature_metadata_path = artifacts_dir / "feature_metadata.pkl"
with open(feature_metadata_path, 'wb') as f:
    pickle.dump(feature_metadata, f)

# Save predictions
predictions_df = pd.DataFrame({
    'y_test_true_log': y_test_log,
    'y_test_pred_log': lgb_test_refined_pred,
    'y_test_true_ms': np.exp(y_test_log),
    'y_test_pred_ms': np.exp(lgb_test_refined_pred),
})

predictions_path = artifacts_dir / "test_predictions.csv"
predictions_df.to_csv(predictions_path, index=False)

# Save feature importance
importance_df.to_csv(artifacts_dir / "feature_importance.csv", index=False)

# Save summary metrics
summary_metrics = {
    'model_type': 'LightGBM (refined features)',
    'test_within_10pct': metrics_lgb.get("lgbtest", {}).get("within_10pct"),
    'test_within_25pct': metrics_lgb.get("lgbtest", {}).get("within_25pct"),
    'test_within_50pct': metrics_lgb.get("lgbtest", {}).get("within_50pct"),
    'test_within_2x': metrics_lgb.get("lgbtest", {}).get("within_2x"),
    'test_rmse_log': metrics_lgb.get("lgbtest", {}).get("rmse_log"),
    'test_mae_ms': metrics_lgb.get("lgbtest", {}).get("mae_ms"),
}

print(f"\nArtifacts saved to: {artifacts_dir}")
print(f"  - Model: {model_path.name}")
print(f"  - Preprocessor: {preprocessor_path.name}")
print(f"  - Feature metadata: {feature_metadata_path.name}")
print(f"  - Predictions: {predictions_path.name}")
print(f"  - Feature importance: {artifacts_dir / 'feature_importance.csv'}")

print("\n" + "="*80)
print("SUMMARY: Best Model Performance on Test Set")
print("="*80)
for key, value in summary_metrics.items():
    print(f"{key:30} {value}")
print("="*80)

## 11. Save Best Model and Prediction Artifacts

In [ ]:
# Extract feature importance from best LightGBM model
feature_importance = lgb_refined.feature_importances_

# Get feature names from preprocessor
def get_feature_names_from_transformer(ct, refined_numeric, categoricals):
    """Extract feature names after one-hot encoding"""
    features = []
    
    # Numeric features
    features.extend(refined_numeric)
    
    # Categorical features (one-hot encoded)
    if hasattr(ct, 'named_transformers_'):
        cat_transformer = ct.named_transformers_.get('cat')
        if cat_transformer is not None:
            onehot = cat_transformer.named_steps.get('onehot')
            if onehot is not None and hasattr(onehot, 'get_feature_names_out'):
                cat_feature_names = onehot.get_feature_names_out(categoricals)
                features.extend(cat_feature_names)
    
    return features

feature_names_refined = get_feature_names_from_transformer(
    preprocess_refined, refined_numeric_features, categorical_features
)

# Create importance dataframe
importance_df = pd.DataFrame({
    'feature': feature_names_refined[:len(feature_importance)],
    'importance': feature_importance
}).sort_values('importance', ascending=False)

# Classify features
importance_df['feature_type'] = importance_df['feature'].apply(
    lambda x: 'Sparse Binary' if '_is_nonzero' in x else 
              'Categorical' if any(x.startswith(f"{cat}_") for cat in categorical_features) else
              'Numeric'
)

print("\nTop 30 Most Important Features:")
print(importance_df.head(30)[['feature', 'importance', 'feature_type']].to_string(index=False))

print("\n\nFeature Importance by Type:")
importance_by_type = importance_df.groupby('feature_type')['importance'].agg(['sum', 'mean', 'count'])
print(importance_by_type)

# Visualize top features
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top 20 features
top_20 = importance_df.head(20)
axes[0].barh(range(len(top_20)), top_20['importance'].values)
axes[0].set_yticks(range(len(top_20)))
axes[0].set_yticklabels(top_20['feature'].values, fontsize=8)
axes[0].set_xlabel('Importance')
axes[0].set_title('Top 20 Most Important Features')
axes[0].invert_yaxis()

# Importance by feature type
type_importance = importance_df.groupby('feature_type')['importance'].sum().sort_values(ascending=False)
axes[1].bar(type_importance.index, type_importance.values, color=['blue', 'orange', 'green'])
axes[1].set_ylabel('Total Importance')
axes[1].set_title('Total Importance by Feature Type')

plt.tight_layout()
plt.show()

## 10. Inspect Feature Importance and Top Contributors

In [ ]:
# Analyze error distribution across target ranges
def analyze_error_by_target_range(y_true, y_pred, split_name=""):
    """Analyze prediction error across different latency ranges"""
    y_true_ms = np.exp(y_true)
    y_pred_ms = np.exp(y_pred)
    
    error_ms = y_pred_ms - y_true_ms
    rel_err = np.abs(error_ms) / y_true_ms
    
    # Define target ranges: low (<1ms), medium (1-10ms), high (>10ms)
    low_mask = y_true_ms < 1
    medium_mask = (y_true_ms >= 1) & (y_true_ms < 10)
    high_mask = y_true_ms >= 10
    
    ranges = [
        ("Low (<1ms)", low_mask),
        ("Medium (1-10ms)", medium_mask),
        ("High (>10ms)", high_mask),
    ]
    
    print(f"\n{split_name} Error by Target Range:")
    print("-" * 80)
    
    for range_name, mask in ranges:
        if mask.sum() > 0:
            mae = np.mean(np.abs(error_ms[mask]))
            mape = np.mean(rel_err[mask]) * 100
            within_10 = np.mean(rel_err[mask] <= 0.10)
            count = mask.sum()
            print(f"{range_name:20} (n={count:5}): MAE={mae:8.4f}ms, MAPE={mape:7.2f}%, within_10%={within_10:.4f}")

# Analyze best model (LightGBM with refined features)
print("=" * 80)
print("BEST MODEL: LightGBM with Refined Features")
print("=" * 80)
analyze_error_by_target_range(y_train_log, lgb_train_refined_pred, "TRAIN")
analyze_error_by_target_range(y_val_log, lgb_val_refined_pred, "VALIDATION")
analyze_error_by_target_range(y_test_log, lgb_test_refined_pred, "TEST")

# Visualize error distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Prediction vs actual (test set)
y_test_true_ms = np.exp(y_test_log)
y_test_pred_ms = np.exp(lgb_test_refined_pred)

axes[0, 0].scatter(y_test_true_ms, y_test_pred_ms, alpha=0.5, s=10)
axes[0, 0].plot([y_test_true_ms.min(), y_test_true_ms.max()], 
                 [y_test_true_ms.min(), y_test_true_ms.max()], 'r--', label='Perfect prediction')
axes[0, 0].set_xlabel('True Latency (ms)')
axes[0, 0].set_ylabel('Predicted Latency (ms)')
axes[0, 0].set_title('Predictions vs Actual (Test Set)')
axes[0, 0].legend()
axes[0, 0].set_xscale('log')
axes[0, 0].set_yscale('log')

# Residuals vs actual
residuals = y_test_pred_ms - y_test_true_ms
axes[0, 1].scatter(y_test_true_ms, residuals, alpha=0.5, s=10)
axes[0, 1].axhline(0, color='r', linestyle='--')
axes[0, 1].set_xlabel('True Latency (ms)')
axes[0, 1].set_ylabel('Prediction Error (ms)')
axes[0, 1].set_title('Residuals vs True Latency')
axes[0, 1].set_xscale('log')

# Relative error distribution
rel_err = np.abs(residuals) / y_test_true_ms
axes[1, 0].hist(rel_err, bins=50, edgecolor='black')
axes[1, 0].axvline(np.median(rel_err), color='r', linestyle='--', label=f'Median: {np.median(rel_err):.3f}')
axes[1, 0].set_xlabel('Relative Error')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Relative Errors')
axes[1, 0].legend()

# Error by log target
axes[1, 1].scatter(y_test_log, rel_err, alpha=0.5, s=10)
axes[1, 1].set_xlabel('Log True Latency')
axes[1, 1].set_ylabel('Relative Error')
axes[1, 1].set_title('Relative Error vs Log Target')
axes[1, 1].axhline(0.1, color='orange', linestyle='--', label='10% threshold')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 9. Evaluate Error by Target Range

In [ ]:
# Create binary presence flags for sparse features (50-95% sparse)
# These capture the zero vs non-zero split that trees naturally exploit
medium_sparse_cols = [col for col in numeric_features 
                      if 0.5 <= sparsity_info[col] < 0.95]

print(f"Creating binary presence flags for {len(medium_sparse_cols)} medium-sparse features")

def add_sparse_flags(X_df, sparse_cols):
    """Add binary presence flags for sparse features"""
    X_with_flags = X_df.copy()
    for col in sparse_cols:
        X_with_flags[f"{col}_is_nonzero"] = (X_df[col] != 0).astype(int)
    return X_with_flags

# Create versions with binary flags
X_train_with_flags = add_sparse_flags(train_df[refined_feature_cols], medium_sparse_cols)
X_val_with_flags = add_sparse_flags(val_df[refined_feature_cols], medium_sparse_cols)
X_test_with_flags = add_sparse_flags(test_df[refined_feature_cols], medium_sparse_cols)

# Update feature cols
refined_numeric_with_flags = refined_numeric_features + [f"{col}_is_nonzero" for col in medium_sparse_cols]
feature_cols_with_flags = refined_numeric_with_flags + categorical_features

print(f"Total features with binary flags: {len(feature_cols_with_flags)}")
print(f"  Numeric: {len(refined_numeric_with_flags)}")
print(f"  Categorical: {len(categorical_features)}")

# Create preprocessing for features with flags
preprocess_with_flags = ColumnTransformer(
    [
        (
            "num",
            SimpleImputer(strategy="median"),
            refined_numeric_with_flags,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            categorical_features,
        ),
    ]
)

preprocess_with_flags.fit(X_train_with_flags)
X_train_flags_processed = preprocess_with_flags.transform(X_train_with_flags)
X_val_flags_processed = preprocess_with_flags.transform(X_val_with_flags)
X_test_flags_processed = preprocess_with_flags.transform(X_test_with_flags)

# Train LightGBM with sparse binary flags
print("\nTraining LightGBM with sparse binary presence flags...")
lgb_with_flags = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_with_flags.fit(X_train_flags_processed, y_train_log)

# Evaluate
lgb_train_flags_pred = lgb_with_flags.predict(X_train_flags_processed)
lgb_val_flags_pred = lgb_with_flags.predict(X_val_flags_processed)
lgb_test_flags_pred = lgb_with_flags.predict(X_test_flags_processed)

print("LightGBM with sparse flags Performance:")
evaluate("Train", y_train_log, lgb_train_flags_pred)
evaluate("Val", y_val_log, lgb_val_flags_pred)
evaluate("Test", y_test_log, lgb_test_flags_pred)

## 8. Add Binary Presence Flags for Sparse Features

In [ ]:
# Create refined feature set: drop extreme sparse and near-constant features
# Drop features with >=99.9% zeros
refined_numeric_features = [col for col in numeric_features if col not in very_sparse_cols]

# Also drop features with zero or near-zero variance
variance_threshold = 1e-6
low_var_cols = []
for col in refined_numeric_features:
    if X_train[col].var() < variance_threshold:
        low_var_cols.append(col)

refined_numeric_features = [col for col in refined_numeric_features if col not in low_var_cols]

print(f"Original numeric features: {len(numeric_features)}")
print(f"After removing >=99.9% sparse: {len(numeric_features) - len(very_sparse_cols)}")
print(f"After removing low-variance: {len(refined_numeric_features)}")
print(f"Features removed: {len(numeric_features) - len(refined_numeric_features)}")

# Create refined features for training
refined_feature_cols = refined_numeric_features + categorical_features
X_train_refined = train_df[refined_feature_cols]
X_val_refined = val_df[refined_feature_cols]
X_test_refined = test_df[refined_feature_cols]

# Create preprocessing for refined features
preprocess_refined = ColumnTransformer(
    [
        (
            "num",
            SimpleImputer(strategy="median"),
            refined_numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            categorical_features,
        ),
    ]
)

preprocess_refined.fit(X_train_refined)
X_train_refined_processed = preprocess_refined.transform(X_train_refined)
X_val_refined_processed = preprocess_refined.transform(X_val_refined)
X_test_refined_processed = preprocess_refined.transform(X_test_refined)

print(f"\nRefined processed features shape: {X_train_refined_processed.shape}")

# Train LightGBM on refined features
print("\nTraining LightGBM on refined features...")
lgb_refined = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_refined.fit(X_train_refined_processed, y_train_log)

# Evaluate refined LightGBM
lgb_train_refined_pred = lgb_refined.predict(X_train_refined_processed)
lgb_val_refined_pred = lgb_refined.predict(X_val_refined_processed)
lgb_test_refined_pred = lgb_refined.predict(X_test_refined_processed)

print("Refined LightGBM Performance:")
evaluate("Train", y_train_log, lgb_train_refined_pred)
evaluate("Val", y_val_log, lgb_val_refined_pred)
evaluate("Test", y_test_log, lgb_test_refined_pred)

## 7. Build Feature Refinement Variants

In [ ]:
# Analyze sparsity by column
sparsity_df = pd.DataFrame({
    'feature': numeric_features,
    'zero_fraction': [sparsity_info[col] for col in numeric_features],
    'nonzero_count': [(X_train[col] != 0).sum() for col in numeric_features],
})
sparsity_df['sparsity_category'] = pd.cut(
    sparsity_df['zero_fraction'],
    bins=[0, 0.5, 0.95, 0.999, 1.0],
    labels=['<50%', '50-95%', '95-99.9%', '>99.9%']
)
sparsity_df = sparsity_df.sort_values('zero_fraction', ascending=False)

print("\nSparsity Distribution:")
print(sparsity_df['sparsity_category'].value_counts().sort_index())

# Visualize sparsity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of sparsity
axes[0].hist(sparsity_df['zero_fraction'], bins=50, edgecolor='black')
axes[0].axvline(0.95, color='r', linestyle='--', label='95% threshold')
axes[0].axvline(0.999, color='orange', linestyle='--', label='99.9% threshold')
axes[0].set_xlabel('Zero Fraction')
axes[0].set_ylabel('Number of Features')
axes[0].set_title('Distribution of Feature Sparsity')
axes[0].legend()

# Box plot by category
sparsity_df.boxplot(column='zero_fraction', by='sparsity_category', ax=axes[1])
axes[1].set_xlabel('Sparsity Category')
axes[1].set_ylabel('Zero Fraction')
axes[1].set_title('Sparsity by Category')
plt.tight_layout()
plt.show()

print("\nTop 20 sparsest features:")
print(sparsity_df[['feature', 'zero_fraction', 'nonzero_count']].head(20).to_string())

## 6. Analyze Sparse Features and Missingness

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame({
    "RF_within_10pct": [metrics_rf.get("train", {}).get("within_10pct", np.nan),
                        metrics_rf.get("val", {}).get("within_10pct", np.nan),
                        metrics_rf.get("test", {}).get("within_10pct", 0.7349)],
    "LGB_within_10pct": [metrics_lgb.get("lgbtrain", {}).get("within_10pct", np.nan),
                         metrics_lgb.get("lgbval", {}).get("within_10pct", np.nan),
                         metrics_lgb.get("lgbtest", {}).get("within_10pct", np.nan)],
    "XGB_within_10pct": [metrics_xgb.get("xgbtrain", {}).get("within_10pct", np.nan),
                         metrics_xgb.get("xgbval", {}).get("within_10pct", np.nan),
                         metrics_xgb.get("xgbtest", {}).get("within_10pct", np.nan)],
})

print("\n\n=== TEST SET COMPARISON ===")
print("within_10pct  | within_25pct  | within_50pct  | within_2x")
print("=" * 60)

models = [
    ("Random Forest", metrics_rf.get("test", {})),
    ("LightGBM", metrics_lgb.get("lgbtest", {})),
    ("XGBoost", metrics_xgb.get("xgbtest", {})),
]

for name, metrics in models:
    within_10 = metrics.get("within_10pct", np.nan)
    within_25 = metrics.get("within_25pct", np.nan)
    within_50 = metrics.get("within_50pct", np.nan)
    within_2x = metrics.get("within_2x", np.nan)
    print(f"{name:20} | {within_10:0.4f}   | {within_25:0.4f}   | {within_50:0.4f}   | {within_2x:0.4f}")

In [ ]:
# Train Random Forest for comparison
print("Training Random Forest baseline for comparison...")
rf_model = Pipeline(
    [
        ("preprocess", preprocess),
        ("regressor", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1, min_samples_leaf=2)),
    ]
)
rf_model.fit(X_train, y_train_log)

rf_train_pred_log = rf_model.predict(X_train)
rf_val_pred_log = rf_model.predict(X_val)
rf_test_pred_log = rf_model.predict(X_test)

metrics_rf = {}
evaluate("Random Forest Train", y_train_log, rf_train_pred_log, metrics_rf)
evaluate("Random Forest Val", y_val_log, rf_val_pred_log, metrics_rf)
evaluate("Random Forest Test", y_test_log, rf_test_pred_log, metrics_rf)

## 5. Compare Baseline (Random Forest) vs Boosting Performance

In [ ]:
# Train XGBoost on log target
print("\n\nTraining XGBoost on log-transformed target...")
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_processed, y_train_log)

# Evaluate XGBoost
xgb_train_pred_log = xgb_model.predict(X_train_processed)
xgb_val_pred_log = xgb_model.predict(X_val_processed)
xgb_test_pred_log = xgb_model.predict(X_test_processed)

metrics_xgb = {}
evaluate("XGBoost Train", y_train_log, xgb_train_pred_log, metrics_xgb)
evaluate("XGBoost Val", y_val_log, xgb_val_pred_log, metrics_xgb)
evaluate("XGBoost Test", y_test_log, xgb_test_pred_log, metrics_xgb)

In [ ]:
# Train LightGBM on log target
print("Training LightGBM on log-transformed target...")
lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(X_train_processed, y_train_log)

# Evaluate LightGBM
lgb_train_pred_log = lgb_model.predict(X_train_processed)
lgb_val_pred_log = lgb_model.predict(X_val_processed)
lgb_test_pred_log = lgb_model.predict(X_test_processed)

metrics_lgb = {}
evaluate("LightGBM Train", y_train_log, lgb_train_pred_log, metrics_lgb)
evaluate("LightGBM Val", y_val_log, lgb_val_pred_log, metrics_lgb)
evaluate("LightGBM Test", y_test_log, lgb_test_pred_log, metrics_lgb)

## 4. Train Baseline Boosting Models on Log Target

In [ ]:
# Create preprocessing pipeline
preprocess = ColumnTransformer(
    [
        (
            "num",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]
            ),
            categorical_features,
        ),
    ]
)

# Fit preprocessor and transform data
preprocess.fit(X_train)
X_train_processed = preprocess.transform(X_train)
X_val_processed = preprocess.transform(X_val)
X_test_processed = preprocess.transform(X_test)

print(f"Processed features shape: {X_train_processed.shape}")
print(f"This includes one-hot encoded categoricals")

In [ ]:
# Define target and excluded columns (same as baseline)
target_col = "average_ms"
excluded_cols = {"model", "run_id", "repo_file", "repo_link", target_col, "stddev_ms", "min_ms", "max_ms"}
feature_cols = [col for col in train_df.columns if col not in excluded_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Excluded columns: {excluded_cols}")

# Prepare features and targets
X_train = train_df[feature_cols]
X_val = val_df[feature_cols]
X_test = test_df[feature_cols]

# Target in both raw and log forms
y_train_raw = train_df[target_col]
y_val_raw = val_df[target_col]
y_test_raw = test_df[target_col]

y_train_log = np.log(y_train_raw.clip(lower=1e-9))
y_val_log = np.log(y_val_raw.clip(lower=1e-9))
y_test_log = np.log(y_test_raw.clip(lower=1e-9))

# Log1p variant for comparison
y_train_log1p = np.log1p(y_train_raw)
y_val_log1p = np.log1p(y_val_raw)
y_test_log1p = np.log1p(y_test_raw)

# Identify numeric vs categorical
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [col for col in feature_cols if col not in numeric_features]

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

# Analyze sparsity
sparsity_info = {}
for col in numeric_features:
    zero_fraction = (X_train[col] == 0).sum() / len(X_train)
    sparsity_info[col] = zero_fraction

# Count sparse features (≥95% zeros)
sparse_cols = [col for col, sparsity in sparsity_info.items() if sparsity >= 0.95]
very_sparse_cols = [col for col, sparsity in sparsity_info.items() if sparsity >= 0.999]

print(f"\nSparsity Analysis:")
print(f"  Features with ≥95% zeros: {len(sparse_cols)}")
print(f"  Features with ≥99.9% zeros: {len(very_sparse_cols)}")
print(f"  Min sparsity: {min(sparsity_info.values()):.4f}")
print(f"  Max sparsity: {max(sparsity_info.values()):.4f}")

## 3. Prepare Baseline Feature Matrix and Target

In [ ]:
def latency_metrics_from_log(y_log_true, y_log_pred):
    """
    Compute latency metrics from log-space predictions and targets.
    Converts back to milliseconds and computes relative error metrics.
    """
    true_ms = np.exp(y_log_true)
    pred_ms = np.exp(y_log_pred)

    error_ms = pred_ms - true_ms
    rel_err = np.abs(error_ms) / true_ms
    ratio_err = np.maximum(pred_ms / true_ms, true_ms / pred_ms)

    rmse_log = np.sqrt(mean_squared_error(y_log_true, y_log_pred))
    rmse_ms = np.sqrt(np.mean(error_ms ** 2))
    rmse_percent = np.sqrt(np.mean(((error_ms / true_ms) * 100) ** 2))
    mae_ms = mean_absolute_error(true_ms, pred_ms)

    return {
        "rmse_log": rmse_log,
        "rmse_ms": rmse_ms,
        "rmse_percent": rmse_percent,
        "mae_ms": mae_ms,
        "median_relative_error": np.median(rel_err),
        "p90_relative_error": np.percentile(rel_err, 90),
        "p95_relative_error": np.percentile(rel_err, 95),
        "median_percent_error": np.median(rel_err) * 100,
        "p90_percent_error": np.percentile(rel_err, 90) * 100,
        "p95_percent_error": np.percentile(rel_err, 95) * 100,
        "median_ratio_error": np.median(ratio_err),
        "p90_ratio_error": np.percentile(ratio_err, 90),
        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
        "within_2x": np.mean(ratio_err <= 2.0),
    }

def evaluate(split_name, y_true_log, y_pred_log, metrics_dict=None):
    """Evaluate model on a split and print metrics."""
    metrics = latency_metrics_from_log(y_true_log, y_pred_log)
    if metrics_dict is not None:
        metrics_dict[split_name] = metrics
    print(f"\n{split_name.upper()}:")
    for key, value in sorted(metrics.items()):
        print(f"  {key}: {value:.4f}")
    return metrics

## 2. Define Latency Evaluation Metrics

In [ ]:
# Locate the data directory
base_dir = next(
    candidate for candidate in [
        Path.cwd() / "training" / "data",
        Path.cwd().parent / "data",
    ]
    if (candidate / "train_set.csv").exists()
)

# Load datasets
train_df = pd.read_csv(base_dir / "train_set.csv")
val_df = pd.read_csv(base_dir / "val_set.csv")
test_df = pd.read_csv(base_dir / "test_set.csv")

# Display dataset info
print("Dataset shapes:")
print(f"  Train: {train_df.shape}")
print(f"  Val:   {val_df.shape}")
print(f"  Test:  {test_df.shape}")
print(f"\nTotal rows: {train_df.shape[0] + val_df.shape[0] + test_df.shape[0]}")
print(f"Total features: {train_df.shape[1]}")
print(f"\nFirst few rows:")
train_df.head()

## 1. Load Dataset and Validate Feature Schema

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
import lightgbm as lgb
import xgboost as xgb
import pickle
import warnings
warnings.filterwarnings('ignore')

# Boosting and Feature Refinement for ONNX Latency Prediction

This notebook implements the next steps for improving the inference-time prediction model:
1. Train boosting models (LightGBM and XGBoost) with better precision than Random Forest
2. Apply log-transform to the target for improved relative error metrics
3. Refine sparse features: drop near-constant columns and add binary presence flags
4. Analyze error distribution across target ranges to identify biases
5. Extract and compare feature importance across dense and sparse columns